# NB3 — Model Training (revised v2)

**What this notebook does:** fits, on the training set only, the models that NB4/NB5 evaluate:
1. **Logistic regression (full feature set)** — the main, interpretable model.
2. **Gradient-boosted trees (full feature set)** — flexible comparison; also a **calibrated** version.
3. **Three baselines a reviewer will ask for:** age only; age + sex; and the **five predictors of the WHO non-laboratory chart**
   (age, sex, systolic BP, BMI, current tobacco) re-fit on LASI.

Everything is fit for two outcomes: the **primary** `chronic_heart_disease` and the **secondary** `ihd_composite`.

## What changed versus the previous version, and why
| # | Change | Reason |
|---|---|---|
| 1 | Imputation is **inside** each pipeline (`IterativeImputer` for continuous, constant `"Unknown"` level for categoricals). | Previously imputation was done before the split (leakage). Now it is re-learned on training folds only, including inside cross-validation and calibration. |
| 2 | **`class_weight="balanced"` removed.** | It inflated predicted risk from ~3% to ~40% and then required a repair step. Plain models are fit on the true prevalence, so probabilities are meaningful from the start. Class-weighting changes the probability scale, not what the model can learn about ranking; the un-weighted models below reach the same AUC range as the old class-weighted ones. |
| 3 | Tobacco enters as **Never / Former / Current / Unknown** (NB1 fix). | Removes the coding artefact that made "Missing tobacco" the top predictor. |
| 4 | **Household-grouped cross-validation** and calibration folds. | Matches the household-grouped split in NB2. |
| 5 | **Baseline models added** (age only, age+sex, WHO-predictor refit). | Needed to show how much the full model adds over trivially available information. |
| 6 | **Secondary outcome** (`ihd_composite`) trained with the identical pipeline. | The primary outcome is not specific to coronary disease; this is the sensitivity analysis. |
| 7 | GBM is **sigmoid-calibrated** with grouped cross-validation (`CalibratedClassifierCV`); logistic regression is **not** recalibrated. | Un-weighted logistic regression is calibrated in-sample by construction; GBM probabilities are checked in NB4 and the calibrated version is the one reported as "% risk". |

## Assumptions (stated explicitly)
- Predictors: age, systolic/diastolic BP, pulse, BMI (continuous); sex, ever-diagnosed diabetes, tobacco status, vigorous/moderate activity frequency, family history of hypertension/diabetes/heart disease/stroke (categorical).
- **Missing data:** continuous variables are imputed by a single regression-based imputation (not true *multiple* imputation, so the added uncertainty from imputing is not propagated); categorical missing values become an explicit `Unknown` level. Missingness is small (≤ ~1.2% for any predictor) and NB4 repeats the evaluation on complete cases only as a sensitivity check.
- Logistic regression uses scikit-learn's default **L2 penalty (C=1)**, so coefficients are mildly shrunk (not an unpenalised maximum-likelihood fit).
- Linear (log-odds) effects for age/BP/BMI in logistic regression; no interaction terms or splines. The GBM is the check on whether non-linearity matters.
- **No hyper-parameter tuning** (scikit-learn defaults, `random_state=42`). A baseline should be reproducible; tuning on the training folds is a legitimate extension, but the GBM and logistic AUCs were already almost identical, so large gains are unlikely.
- `HistGradientBoostingClassifier` uses its default internal early-stopping split.

## Limitations
- The outcome is **self-reported, prevalent chronic heart disease** — the model estimates the probability that someone *reports having been diagnosed*, not their future risk, and not angiographic CAD.
- Predictors were measured after diagnosis. Coefficients (especially for tobacco status, BP, BMI, activity and diabetes) can be distorted by treatment, lifestyle change after diagnosis and survival, so they must **not** be read as causal risk factors.
- Cross-validated numbers here are for sanity-checking only; the headline numbers come from NB4.
- No lab values, no medication data, no survey weights.

## 0. Environment setup

In [1]:
# ============================================================
# ENVIRONMENT SETUP (same logic as before + one optional override)
# ============================================================
import os

WINDOWS_PATH = r"C:\capstone\capstone_exec\data"
COLAB_DRIVE_PATH = "/content/drive/MyDrive/capstone_exec/data"

# Optional: set the environment variable CAPSTONE_DATA to force a data folder
# (used only so the notebooks can be executed headlessly; ignore it on your own machine).
if os.environ.get("CAPSTONE_DATA"):
    BASE_PATH = os.environ["CAPSTONE_DATA"]
    print("Using CAPSTONE_DATA override.")
elif os.path.exists("/content"):
    print("Detected a genuine hosted Colab runtime (found /content).")
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_PATH = COLAB_DRIVE_PATH
else:
    print("No /content found - treating this as a local machine (VS Code, Jupyter, etc).")
    BASE_PATH = WINDOWS_PATH

if not os.path.exists(BASE_PATH):
    fallback_candidates = [
        r"C:\capstone\capstone_exec\data",
        r"C:\capstone\possible datasets\lasi-1",
        "/content/drive/MyDrive/capstone_exec/data",
        os.path.join(os.getcwd(), "data"),
    ]
    found = next((c for c in fallback_candidates if os.path.exists(c)), None)
    if found:
        print(f"BASE_PATH did not exist, using fallback: {found}")
        BASE_PATH = found
    else:
        raise FileNotFoundError("Could not find the data folder anywhere. Checked:\n  "
                                + "\n  ".join([BASE_PATH] + fallback_candidates))

LASI_RAW = os.path.join(BASE_PATH, "lasi", "raw")
LASI_PROCESSED = os.path.join(BASE_PATH, "lasi", "processed")
MODELS_DIR = os.path.join(BASE_PATH, "..", "outputs", "models")
FIGURES_DIR = os.path.join(BASE_PATH, "..", "outputs", "figures")
TABLES_DIR = os.path.join(BASE_PATH, "..", "outputs", "tables")
for _d in (LASI_PROCESSED, MODELS_DIR, FIGURES_DIR, TABLES_DIR):
    os.makedirs(_d, exist_ok=True)

print("\nBASE_PATH:", BASE_PATH)
print("LASI_PROCESSED:", LASI_PROCESSED)

No /content found - treating this as a local machine (VS Code, Jupyter, etc).

BASE_PATH: C:\capstone\capstone_exec\data
LASI_PROCESSED: C:\capstone\capstone_exec\data\lasi\processed


## 1. Pipeline definitions
The cell below is the single definition of all pipelines. It is written verbatim to `outputs/models/lasi_pipeline_defs.py` as a record (NB4's leave-states-out analysis repeats the same definitions — if you edit one place, edit the other).

In [2]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

def make_preprocessor(cont, cat):
    """Imputation lives INSIDE the pipeline, so it is re-fit on training folds only."""
    parts = []
    if cont:
        imp = IterativeImputer(max_iter=10, random_state=42) if len(cont) > 1 else SimpleImputer(strategy="median")
        parts.append(("continuous", Pipeline([("impute", imp), ("scale", StandardScaler())]), cont))
    if cat:
        parts.append(("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat))
    return ColumnTransformer(parts)

def make_lr(cont, cat):
    return Pipeline([("pre", make_preprocessor(cont, cat)),
                     ("model", LogisticRegression(max_iter=3000))])

def make_gbm(cont, cat):
    return Pipeline([("pre", make_preprocessor(cont, cat)),
                     ("model", HistGradientBoostingClassifier(random_state=42))])


# Save a copy of these definitions next to the models, for provenance.
import inspect, sys
_src = r"""import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

def make_preprocessor(cont, cat):
    '''Imputation lives INSIDE the pipeline, so it is re-fit on training folds only.'''
    parts = []
    if cont:
        imp = IterativeImputer(max_iter=10, random_state=42) if len(cont) > 1 else SimpleImputer(strategy="median")
        parts.append(("continuous", Pipeline([("impute", imp), ("scale", StandardScaler())]), cont))
    if cat:
        parts.append(("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat))
    return ColumnTransformer(parts)

def make_lr(cont, cat):
    return Pipeline([("pre", make_preprocessor(cont, cat)),
                     ("model", LogisticRegression(max_iter=3000))])

def make_gbm(cont, cat):
    return Pipeline([("pre", make_preprocessor(cont, cat)),
                     ("model", HistGradientBoostingClassifier(random_state=42))])
"""
with open(os.path.join(MODELS_DIR, "lasi_pipeline_defs.py"), "w") as f:
    f.write(_src)
print("Pipeline definitions loaded and saved.")

Pipeline definitions loaded and saved.


## 2. Load the training set and define feature sets

In [3]:
import json, joblib
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

train_df = pd.read_csv(os.path.join(LASI_PROCESSED, "lasi_train.csv"), dtype={"state_code": str})
print("Training set:", train_df.shape)

CONTINUOUS = ["age_years", "average_systolic_bp_mmHg", "average_diastolic_bp_mmHg", "average_pulse_bpm", "BMI"]
CATEGORICAL = ["sex", "diabetes", "tobacco_status", "vigorous_activity_frequency", "moderate_activity_frequency",
               "family_history_hypertension_any", "family_history_diabetes_any",
               "family_history_heart_disease_any", "family_history_stroke_any"]

FEATURE_SETS = {
    "full":           (CONTINUOUS, CATEGORICAL),
    "age_only":       (["age_years"], []),
    "age_sex":        (["age_years"], ["sex"]),
    "who_predictors": (["age_years", "average_systolic_bp_mmHg", "BMI"], ["sex", "current_tobacco_any"]),
}
OUTCOMES = {"chd": "chronic_heart_disease", "ihd": "ihd_composite"}

with open(os.path.join(MODELS_DIR, "feature_list.json"), "w") as f:
    json.dump({"continuous": CONTINUOUS, "categorical": CATEGORICAL,
               "feature_sets": {k: {"continuous": v[0], "categorical": v[1]} for k, v in FEATURE_SETS.items()},
               "outcomes": OUTCOMES}, f, indent=2)
print("Saved feature_list.json")
for k, (c, g) in FEATURE_SETS.items():
    print(f"  {k:15s}: {len(c)+len(g)} predictors -> {c + g}")

Training set: (46748, 22)
Saved feature_list.json
  full           : 14 predictors -> ['age_years', 'average_systolic_bp_mmHg', 'average_diastolic_bp_mmHg', 'average_pulse_bpm', 'BMI', 'sex', 'diabetes', 'tobacco_status', 'vigorous_activity_frequency', 'moderate_activity_frequency', 'family_history_hypertension_any', 'family_history_diabetes_any', 'family_history_heart_disease_any', 'family_history_stroke_any']
  age_only       : 1 predictors -> ['age_years']
  age_sex        : 2 predictors -> ['age_years', 'sex']
  who_predictors : 5 predictors -> ['age_years', 'average_systolic_bp_mmHg', 'BMI', 'sex', 'current_tobacco_any']


## 3. Fit all models for both outcomes
For each outcome: drop training rows whose outcome is unknown, build **household-grouped, outcome-stratified 5-fold** splits, fit the six models, and compute a grouped cross-validated AUC for sanity-checking.

The **calibrated GBM** is fit by `CalibratedClassifierCV(method="sigmoid")` using those same grouped folds (the base GBM is fit on 4 folds and the sigmoid is learned on the 5th, then the 5 resulting models are averaged). No test data is involved.

In [4]:
import warnings; warnings.filterwarnings("ignore")
fitted = {}      # (outcome_key, model_name) -> fitted estimator
cv_rows = []

for okey, ocol in OUTCOMES.items():
    d = train_df.dropna(subset=[ocol]).reset_index(drop=True)
    y = (d[ocol] == "Yes").astype(int).values
    groups = d["household_id"].values
    splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(d, y, groups))
    print(f"\n=== Outcome: {ocol}  (n={len(d)}, positives={y.sum()}, prevalence={y.mean():.3%}) ===")

    specs = {
        "logistic_full":        ("lr",  "full"),
        "gbm_full":             ("gbm", "full"),
        "baseline_age_only":    ("lr",  "age_only"),
        "baseline_age_sex":     ("lr",  "age_sex"),
        "baseline_who_refit":   ("lr",  "who_predictors"),
    }
    for name, (kind, fs) in specs.items():
        cont, cat = FEATURE_SETS[fs]
        est = (make_lr if kind == "lr" else make_gbm)(cont, cat)
        X = d[cont + cat]
        scores = cross_val_score(est, X, y, cv=splits, scoring="roc_auc")
        est.fit(X, y)
        fitted[(okey, name)] = est
        cv_rows.append({"outcome": ocol, "model": name, "cv_auc_mean": scores.mean(), "cv_auc_sd": scores.std(),
                        "train_mean_pred_risk": est.predict_proba(X)[:, 1].mean(), "train_prevalence": y.mean()})
        print(f"  {name:20s} grouped 5-fold CV AUC = {scores.mean():.3f} (sd {scores.std():.3f})")

    cont, cat = FEATURE_SETS["full"]
    cal = CalibratedClassifierCV(make_gbm(cont, cat), method="sigmoid", cv=splits)
    cal.fit(d[cont + cat], y)
    fitted[(okey, "gbm_calibrated")] = cal
    print(f"  {'gbm_calibrated':20s} fitted (sigmoid, grouped folds); train mean predicted risk = "
          f"{cal.predict_proba(d[cont + cat])[:, 1].mean():.3%} vs prevalence {y.mean():.3%}")

cv_table = pd.DataFrame(cv_rows)
print("\nSummary:")
print(cv_table.round(4).to_string(index=False))


=== Outcome: chronic_heart_disease  (n=46748, positives=1586, prevalence=3.393%) ===
  logistic_full        grouped 5-fold CV AUC = 0.740 (sd 0.011)
  gbm_full             grouped 5-fold CV AUC = 0.732 (sd 0.012)
  baseline_age_only    grouped 5-fold CV AUC = 0.646 (sd 0.009)
  baseline_age_sex     grouped 5-fold CV AUC = 0.648 (sd 0.007)
  baseline_who_refit   grouped 5-fold CV AUC = 0.684 (sd 0.012)
  gbm_calibrated       fitted (sigmoid, grouped folds); train mean predicted risk = 3.418% vs prevalence 3.393%

=== Outcome: ihd_composite  (n=46680, positives=1047, prevalence=2.243%) ===
  logistic_full        grouped 5-fold CV AUC = 0.762 (sd 0.010)
  gbm_full             grouped 5-fold CV AUC = 0.744 (sd 0.013)
  baseline_age_only    grouped 5-fold CV AUC = 0.657 (sd 0.014)
  baseline_age_sex     grouped 5-fold CV AUC = 0.664 (sd 0.017)
  baseline_who_refit   grouped 5-fold CV AUC = 0.707 (sd 0.013)
  gbm_calibrated       fitted (sigmoid, grouped folds); train mean predicted risk = 

### How to read this output
- All CV AUCs must be clearly above 0.5 and fold-to-fold sd should be small (a few thousandths to ~0.02).
- Expect an **ordering**: age-only < age+sex < WHO-predictor refit < full models. The *size* of the gap between the baselines and the full model is the honest measure of what the extra variables (diabetes, family history, activity, diastolic BP, pulse, tobacco status) buy. If it is small, say so in the paper rather than emphasising the AUC of the full model alone.
- Logistic and GBM AUCs being nearly identical means there is no strong non-linear/interaction signal in these 14 predictors.
- `train_mean_pred_risk` should be close to `train_prevalence` for all un-weighted models (a property of fitting on the true class balance, not a performance claim).

## 4. Save the fitted models

In [5]:
for (okey, name), est in fitted.items():
    joblib.dump(est, os.path.join(MODELS_DIR, f"{okey}__{name}.joblib"))
cv_table.to_csv(os.path.join(TABLES_DIR, "training_cv_summary.csv"), index=False)
print("Saved", len(fitted), "models to", MODELS_DIR)
print(sorted(f"{k[0]}__{k[1]}" for k in fitted))

Saved 12 models to C:\capstone\capstone_exec\data\..\outputs\models
['chd__baseline_age_only', 'chd__baseline_age_sex', 'chd__baseline_who_refit', 'chd__gbm_calibrated', 'chd__gbm_full', 'chd__logistic_full', 'ihd__baseline_age_only', 'ihd__baseline_age_sex', 'ihd__baseline_who_refit', 'ihd__gbm_calibrated', 'ihd__gbm_full', 'ihd__logistic_full']


## Summary
Twelve fitted models (six per outcome) and `feature_list.json`. The **calibrated GBM** and the **un-weighted logistic regression** are the models whose probabilities may be described as "% risk" — and only as the *probability of reporting a prior diagnosis* in a LASI-like sample.

**Next:** `nb4_lasi_model_evaluation.ipynb` (held-out test set, calibration, decision curves, subgroup/sensitivity analyses, leave-states-out validation).